# MB51 — Exploración interactiva PP04
**Clases:** 261 (orden de producción) · 201 (centro de coste)  
**Período:** 2019 → 2026 

> Ejecuta **Celda 0** primero (carga ~10 s). El resto responde en tiempo real.

## 0 · Carga y limpieza (ejecutar una vez)

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as w
from IPython.display import display, clear_output
from pathlib import Path

PARQUET = Path('../data/mb51.parquet')
assert PARQUET.exists(), 'Ejecuta primero scripts/descarga.py'

def parse_sap_num(serie):
    s = serie.str.strip()
    negativo = s.str.endswith('-')
    valor = pd.to_numeric(s.str.rstrip('-').str.replace(',', '', regex=False), errors='coerce')
    return valor.where(~negativo, -valor)

print('Cargando...')
df = pd.read_parquet(PARQUET)
df['CANTIDAD_N'] = parse_sap_num(df['CANTIDAD'])
df['IMPORTE_N']  = parse_sap_num(df['IMPORTE'])
df['FECHA']      = pd.to_datetime(df['REGISTRADO_EL'], format='%Y%m%d', errors='coerce')
df['AÑO']        = df['FECHA'].dt.year.astype('Int64')
df['MES']        = df['FECHA'].dt.to_period('M')

AÑOS      = sorted(df['AÑO'].dropna().unique().tolist())
MATERIALES = sorted(df['TEXTO_MATE'].dropna().unique().tolist())








# Identificar lotes de la familia Y3 (Herencia por lote)
LOTES_Y3 = set(df[df['TEXTO_MATE'].str.contains('y3', case=False, na=False)]['LOTE'].unique())

# Cargar órdenes preempaquetadas si existen
PATH_PREEMPQ = Path('../data/preempaque.parquet')
df_preempq = pd.read_parquet(PATH_PREEMPQ) if PATH_PREEMPQ.exists() else pd.DataFrame(columns=['ORDEN'])
ORDENES_PREEMPQ = set(df_preempq['ORDEN'].unique())
if not df_preempq.empty:
    print(f'OK — {len(df):,} filas cargadas. Años: {AÑOS[0]}–{AÑOS[-1]}  |  {len(ORDENES_PREEMPQ):,} órdenes PREEMPAQUE.')
else:
    print(f'OK — {len(df):,} filas cargadas. Años: {AÑOS[0]}–{AÑOS[-1]}  |  AVISO: No se encontró data/preempaque.parquet')


Cargando...
OK — 9,504,194 filas cargadas. Años disponibles: 2019–2026


## 1 · Explorador de movimientos
Agrupa y filtra los movimientos. Usa los controles para cambiar la vista.

In [3]:
# ── Controles (se crean una sola vez) ──────────────────────────────────
if 'w_clase' not in globals():
    w_clase = w.SelectMultiple(
        options=['261', '201'], value=['261', '201'],
        description='Clase mov:', rows=2
    )
    w_años = w.IntRangeSlider(
        value=[AÑOS[0], AÑOS[-1]], min=AÑOS[0], max=AÑOS[-1], step=1,
        description='Años:', continuous_update=False, layout=w.Layout(width='500px')
    )
    w_texto = w.Text(
        value='', placeholder='ej: y3, pvb, modmed ...',
        description='Material:'
    )
    w_agrupar = w.Dropdown(
        options=['TEXTO_MATE', 'MATERIAL', 'AÑO', 'MES', 'CLASE_MOV', 'LOTE', 'ORDEN'],
        value='TEXTO_MATE', description='Agrupar por:'
    )
    w_metrica = w.Dropdown(
        options=[('M2 consumidos', 'm2'), ('Importe (GCOP)', 'gcop'), ('Movimientos', 'mov')],
        value='m2', description='Metrica:'
    )
    w_top = w.IntSlider(value=20, min=5, max=50, step=5, description='Top N:')
    w_grafico = w.Checkbox(value=True, description='Mostrar grafico')
    out = w.Output()

def actualizar(_=None):
    with out:
        clear_output(wait=True)
        # Filtros
        mask = (
            df['CLASE_MOV'].isin(list(w_clase.value)) &
            df['AÑO'].between(w_años.value[0], w_años.value[1])
        )
        if w_texto.value.strip():
            mask &= df['TEXTO_MATE'].str.contains(w_texto.value.strip(), case=False, na=False)
        sub = df[mask]
        if sub.empty:
            print('Sin resultados con los filtros actuales.')
            return
        # Agrupación
        grp_col = w_agrupar.value
        grp = sub.groupby(grp_col, observed=True).agg(
            movimientos  =('ID',         'count'),
            m2           =('CANTIDAD_N', lambda x: x.abs().sum()),
            gcop         =('IMPORTE_N',  lambda x: x.abs().sum() / 1e9),
            ordenes      =('ORDEN',      'nunique'),
            lotes        =('LOTE',       'nunique'),
        ).sort_values(w_metrica.value, ascending=False).head(w_top.value)
        # Tabla
        print(f'Registros filtrados: {len(sub):,}  |  Grupos: {len(grp)}')
        display(grp.round(4))
        # Grafico
        if w_grafico.value and len(grp) > 0:
            etiquetas = grp.index.astype(str)
            valores   = grp[w_metrica.value]
            fig, ax = plt.subplots(figsize=(12, max(4, len(grp) * 0.35)))
            ax.barh(etiquetas, valores)
            ax.invert_yaxis()
            unidad_label = {'m2': 'M2', 'gcop': 'GCOP', 'mov': 'movimientos'}[w_metrica.value]
            ax.set_xlabel(unidad_label)
            ax.set_title(f'Top {w_top.value} por {grp_col} — {unidad_label}')
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}'))
            plt.tight_layout()
            plt.show()

for widget in [w_clase, w_años, w_texto, w_agrupar, w_metrica, w_top, w_grafico]:
    widget.unobserve_all()
    widget.observe(actualizar, names='value')

panel = w.VBox([
    w.HBox([w_clase, w_años]),
    w.HBox([w_texto, w_agrupar, w_metrica]),
    w.HBox([w_top, w_grafico]),
    out
])
actualizar()
display(panel)

## 2 · Lotes por material Y3
Enfocado en los 4 materiales Y3. Selecciona el material y el rango de años.

In [9]:
# Ahora df_y3 incluye materiales vinculados por LOTE (ej. MODMED)
df_y3 = df[df['LOTE'].isin(LOTES_Y3)].copy()
MATS_Y3 = sorted(df_y3['TEXTO_MATE'].unique().tolist())
ALMACENES_Y3 = sorted(df_y3['ALMACEN'].dropna().unique().tolist())

# Controles (se crean una sola vez)
if 'w_mat_y3' not in globals():
    w_almacen_y3 = w.SelectMultiple(
        options=ALMACENES_Y3, value=['PP04'],
        description='Almacen:', rows=4, layout=w.Layout(width='200px')
    )
    w_mat_y3 = w.SelectMultiple(
        options=MATS_Y3, value=MATS_Y3,
        description='Material Y3:', rows=4, layout=w.Layout(width='300px')
    )
    w_años_y3 = w.IntRangeSlider(
        value=[AÑOS[0], AÑOS[-1]], min=AÑOS[0], max=AÑOS[-1], step=1,
        description='Años:', continuous_update=False, layout=w.Layout(width='500px')
    )
    w_agrupar_y3 = w.Dropdown(
        options=['LOTE', 'ORDEN', 'AÑO', 'MES', 'TEXTO_MATE'],
        value='LOTE', description='Agrupar por:'
    )
    w_metrica_y3 = w.Dropdown(
        options=[('M2 consumidos', 'm2'), ('Importe (GCOP)', 'gcop'), ('Movimientos', 'mov')],
        value='m2', description='Metrica:'
    )
    w_top_y3 = w.IntSlider(value=20, min=5, max=100, step=5, description='Top N:')
    out_y3 = w.Output()

def actualizar_y3(_=None):
    with out_y3:
        clear_output(wait=True)
        mask = (
            df_y3['TEXTO_MATE'].isin(list(w_mat_y3.value)) &
            df_y3['AÑO'].between(w_años_y3.value[0], w_años_y3.value[1]) &
            df_y3['ALMACEN'].isin(list(w_almacen_y3.value))
        )
        sub = df_y3[mask]
        if sub.empty:
            print('Sin resultados.')
            return
        grp_col = w_agrupar_y3.value
        grp = sub.groupby(grp_col, observed=True).agg(
            movimientos=('ID',         'count'),
            m2         =('CANTIDAD_N', lambda x: x.abs().sum()),
            gcop       =('IMPORTE_N',  lambda x: x.abs().sum() / 1e9),
            ordenes    =('ORDEN',      'nunique'),
        ).sort_values(w_metrica_y3.value, ascending=False).head(w_top_y3.value)
        print(f'Registros filtrados: {len(sub):,}  |  Grupos: {len(grp)}')
        display(grp.round(4))

for widget in [w_mat_y3, w_años_y3, w_agrupar_y3, w_metrica_y3, w_top_y3, w_almacen_y3]:
    widget.unobserve_all()
    widget.observe(actualizar_y3, names='value')

panel_y3 = w.VBox([
    w.HBox([w_mat_y3, w_almacen_y3, w_años_y3]),
    w.HBox([w_agrupar_y3, w_metrica_y3, w_top_y3]),
    out_y3
])
actualizar_y3()
display(panel_y3)

## 3 · Detalle de una orden
Ingresa un número de orden para ver todos los materiales que consumió (receta real).

In [5]:
# Detalle de una orden
if 'w_orden' not in globals():
    w_orden = w.Text(
        value='', placeholder='ej: 000011265325',
        description='Orden:', layout=w.Layout(width='350px')
    )
    out_ord = w.Output()

def buscar_orden(_=None):
    with out_ord:
        clear_output(wait=True)
        orden = w_orden.value.strip().zfill(12)
        if not orden or orden == '000000000000':
            return
        
        sub = df[df['ORDEN'] == orden]
        if sub.empty:
            print(f'Orden {orden} no encontrada en los datos de MB51.')
            return
        
        is_preempq = orden in ORDENES_PREEMPQ
        status_label = '✅ PREEMPAQUE' if is_preempq else '⏳ PENDIENTE / EN PROCESO'
        print(f'Orden: {orden}  |  Estado: {status_label}')
        print(f'Fecha: {sub["FECHA"].min().date()} → {sub["FECHA"].max().date()}')
        
        receta = (
            sub.groupby(['MATERIAL', 'TEXTO_MATE', 'LOTE', 'UNIDAD'])
            .agg(
                m2       =('CANTIDAD_N', lambda x: x.abs().sum()),
                lineas   =('ID',         'count'),
            )
            .reset_index()
            .sort_values('m2', ascending=False)
        )
        receta['es_y3'] = receta['TEXTO_MATE'].str.contains('y3', case=False, na=False)
        display(receta)

w_orden.unobserve_all()
w_orden.observe(buscar_orden, names='value')
display(w.VBox([w_orden, out_ord]))


## 4 · Análisis ad-hoc
`df` tiene `CANTIDAD_N`, `IMPORTE_N`, `FECHA`, `AÑO`, `MES` listos.

In [15]:
# Ejemplo: consumo mensual de Y3 por tipo
(
    df[df['TEXTO_MATE'].str.contains('y3', case=False, na=False)]
    .groupby(['MES', 'TEXTO_MATE'])
    .agg(m2=('CANTIDAD_N', lambda x: x.abs().sum()))
    .reset_index()
    .pivot(index='MES', columns='TEXTO_MATE', values='m2')
    .fillna(0)
    .tail(24)  # últimos 24 meses
)

TEXTO_MATE,0000 PROBETA R&DC L19-31PROB-Y3,0000 PROBETA R&DC L19-31PROBY3-NB3,LAT DEL DER R&DC LDD-L21-1BY3-DURABILIDA,LAT DEL DER R&DC LDD-L21-1Y3-DURABILIDAD,LAT DEL IZQ R&DC LDI-L21-1BY3-DURABILIDA,LAT DEL IZQ R&DC LDI-L21-1Y3-DURABILIDAD,PU Y3 0.63 X1016,PU Y3 0.63 X1524,PU Y3 1.27 X1016,PU Y3 1.27 X1524,TECHO R&DC TECHO-L21-1-Y3-DURABILIDAD
MES,,,,,,,,,,,
2022-03,0.0,24.0,0.0,0.0,0.0,0.0,0.0,13744.480,0.000,397.634,0.0
2022-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12646.961,0.000,438.526,0.0
2022-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15838.611,0.000,446.166,0.0
2022-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15853.565,0.000,6210.158,0.0
2022-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16252.853,0.000,5553.920,0.0
2022-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,764.481,0.000,297.551,0.0
2022-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22097.511,0.000,283.387,0.0
2022-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,90715.889,0.000,400.745,0.0
2022-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12744.066,0.000,72.496,0.0


## 5 · Búsqueda del Ciclo de Vida de un Lote
Ingresa el número de lote para auditar transversalmente qué materiales y movimientos tuvo (útil para ver la equivalencia Directo vs MODMED).

In [ ]:
w_lote_buscar = w.Text(
    value='', placeholder='ej: 0001859663',
    description='Buscar Lote:', layout=w.Layout(width='350px')
)
out_lote = w.Output()

def buscar_lote(_=None):
    with out_lote:
        clear_output(wait=True)
        lote = w_lote_buscar.value.strip()
        if not lote:
            return

        # ── Identificar origen del lote en IM02 ─────────────────────
        origen_im02 = df[(df['LOTE'] == lote) & (df['ALMACEN'] == 'IM02')]
        if not origen_im02.empty:
            mats_origen = origen_im02['TEXTO_MATE'].value_counts()
            print(f'LOTE: {lote}')
            print(f'\nOrigen en IM02 (identificación del material):')
            for mat, cnt in mats_origen.head(5).items():
                print(f'  {mat}  ({cnt} registros)')
            is_y3 = origen_im02['TEXTO_MATE'].str.contains('y3', case=False, na=False).any()
            print(f'  → Es lote Y3: {"✅ SI" if is_y3 else "❌ No"}')
        else:
            print(f'LOTE: {lote}')
            print(f'  No tiene registros en IM02')

        # ── Movimientos de consumo (261/201) ────────────────────────
        sub = df[(df['LOTE'] == lote) & (df['CLASE_MOV'].isin(['261', '201']))]
        if sub.empty:
            print(f'\nNo registra movimientos de consumo.')
            return

        mats_consumo = sub['TEXTO_MATE'].unique().tolist()
        m2_tot = sub['CANTIDAD_N'].abs().sum()
        imp_tot = sub['IMPORTE_N'].abs().sum()

        print(f'\nMovimientos de consumo:')
        print(f'  Materiales: {mats_consumo}')
        print(f'  Movimientos: {len(sub):,}  |  Órdenes: {sub["ORDEN"].nunique():,}')
        print(f'  Consumo: {m2_tot:,.2f} M2  |  Importe: {imp_tot:,.0f} COP')
        print(f'  Almacenes: {sub["ALMACEN"].value_counts().to_dict()}')
        
        display(sub[['ID', 'FECHA', 'TEXTO_MATE', 'ORDEN', 'CLASE_MOV', 'ALMACEN', 'CANTIDAD_N', 'IMPORTE_N']].sort_values('FECHA', ascending=False).head(30))

w_lote_buscar.unobserve_all()
w_lote_buscar.observe(buscar_lote, names='value')
display(w.VBox([w_lote_buscar, out_lote]))


## 6 · Composición entre último cristal y PC
Filtra ZFER según las capas que hay entre el último vidrio (SL) y el policarbonato (PC).
Ideal para encontrar la estructura Cristal → Y3 → Y3 → MODMED → PC.

In [ ]:
# Cargar la hoja de composición
import pandas as pd
from pathlib import Path

PATH_COMPOSICION = Path('../outputs/recetas_y3.xlsx')

if not PATH_COMPOSICION.exists():
    print('Ejecuta primero: python scripts/recetas_y3.py')
else:
    df_comp = pd.read_excel(PATH_COMPOSICION, sheet_name='composicion_sl_pc')
    print(f'ZFER con composición SL→PC: {len(df_comp)}')
    print(f'Composiciones únicas: {df_comp["composicion"].nunique()}')
    print('\nTop composiciones:')
    print(df_comp['composicion'].value_counts())

    # Widget de filtrado
    composiciones = sorted(df_comp['composicion'].unique().tolist())

    w_comp = w.SelectMultiple(
        options=composiciones, value=composiciones,
        description='Composicion:', rows=6, layout=w.Layout(width='400px')
    )
    out_comp = w.Output()

    def filtrar_comp(_=None):
        with out_comp:
            clear_output(wait=True)
            sub = df_comp[df_comp['composicion'].isin(list(w_comp.value))]
            if sub.empty:
                print('Sin resultados.')
                return
            display(sub[['ZFER_MATERIAL', 'TEXTO_BREVE_MATERIAL', 'ZFOR',
                         'ultimo_SL_pos', 'PC_pos', 'n_y3', 'n_modmed',
                         'composicion', 'capas_entre']].sort_values('composicion'))

    w_comp.unobserve_all()
    w_comp.observe(filtrar_comp, names='value')

    display(w.VBox([w_comp, out_comp]))
    filtrar_comp()
